In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load and Combine Datasets (Minimum 6 months)
data_folder = "./data"
file_list = sorted(glob.glob(os.path.join(data_folder, "CRMLSSold*.csv")))

if len(file_list) < 6:
    print(f"Warning: Found only {len(file_list)} files. Ensure you have downloaded at least 6 months of data.")

df_raw = pd.concat([pd.read_csv(f, low_memory=False) for f in file_list], ignore_index=True)


In [40]:
# 2. Domain Task Filter: Residential Single Family Homes Only
df_sfr = df_raw[
    (df_raw['PropertyType'] == 'Residential') & 
    (df_raw['PropertySubType'] == 'SingleFamilyResidence')
].copy()

df_sfr['CloseDate'] = pd.to_datetime(df_sfr['CloseDate'])


In [41]:
# 3. Structural Impossibilities & Invalid Records (Cleaned DataFrame retained)
df_clean = df_sfr[
    (df_sfr['ClosePrice'] > 0) & 
    (df_sfr['LivingArea'] > 0) & 
    (df_sfr['LotSizeSquareFeet'] > 0) &
    (df_sfr['LotSizeSquareFeet'] >= df_sfr['LivingArea'])  # Fixed inequality
].copy()


In [42]:
# 4. Leakage Elimination
leakage_cols = [
    'ListPrice', 'OriginalListPrice', 'PreviousListPrice', 
    'DaysOnMarket', 'CumulativeDaysOnMarket', 'PriceChangeTimestamp',
    'MajorChangeType', 'ConcessionsAmount', 'BuyerBrokerageCompensation'
]
df_clean = df_clean.drop(columns=[c for c in leakage_cols if c in df_clean.columns])


In [43]:
#5. Missing value analysis

# Total missing values per column
missing_count = df_clean.isnull().sum()

# Percentage of missing values per column
missing_percent = (df_clean.isnull().sum() / len(df_clean)) * 100

# Summary table of columns with missing values
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Percent Missing (%)': missing_percent
}).query('`Missing Count` > 0').sort_values(by='Missing Count', ascending=False)

print(missing_summary)
print(missing_summary.iloc[0:30])

                              Missing Count  Percent Missing (%)
AboveGradeFinishedArea               301480           100.000000
TaxAnnualAmount                      301480           100.000000
FireplacesTotal                      301480           100.000000
MiddleOrJuniorSchoolDistrict         301480           100.000000
BusinessType                         301480           100.000000
...                                     ...                  ...
BuyerAgentLastName                      126             0.041794
ListAgentFullName                        58             0.019238
BathroomsTotalInteger                    49             0.016253
ListAgentLastName                        27             0.008956
PostalCode                                1             0.000332

[64 rows x 2 columns]
                              Missing Count  Percent Missing (%)
AboveGradeFinishedArea               301480           100.000000
TaxAnnualAmount                      301480           100.000000
Fi

In [44]:
cols_100_missing = [
    "AboveGradeFinishedArea",
    "TaxAnnualAmount",
    "FireplacesTotal",
    "MiddleOrJuniorSchoolDistrict",
    "BusinessType",
    "CoveredSpaces",
    "TaxYear",
    "ElementarySchoolDistrict",
]
df_clean.drop(columns=[c for c in cols_100_missing if c in df_clean.columns], inplace=True)

other = [
    "latfilled", # not useful
    "lonfilled", # not useful
    "CoBuyerAgentFirstName" # leakage
]
df_clean.drop(columns=[c for c in other if c in df_clean.columns], inplace=True)

# Drop high-missingness raw identifier/string columns to remove noise
high_missing_cols = [
    "CoBuyerAgentFirstName", # leakage
    "CoListAgentFirstName", # leakage
    "CoListAgentLastName", # leakage
    "CoListOfficeName", # leakage
    "BuyerAgencyCompensation", # leakage
    "BuyerAgencyCompensationType", # leakage
    "BuilderName",
    "LotSizeDimensions",
    "BuildingAreaTotal",
    "OriginatingSystemName",
    "OriginatingSystemSubName",
]
df_clean.drop(columns=[c for c in high_missing_cols if c in df_clean.columns], inplace=True)

# keeping school districts for now but should find them by coordinating with school map or something


In [45]:
# Impute implicit negatives/zeros
if "WaterfrontYN" in df_clean.columns:
    df_clean["WaterfrontYN"] = df_clean["WaterfrontYN"].fillna(0).astype(int)
if "BasementYN" in df_clean.columns:
    df_clean["BasementYN"] = df_clean["BasementYN"].fillna(0).astype(int)
if "BelowGradeFinishedArea" in df_clean.columns:
    df_clean["BelowGradeFinishedArea"] = df_clean["BelowGradeFinishedArea"].fillna(0.0)

if "AssociationFee" in df_clean.columns:
    # Explicit binary missing indicator for HOA flag
    df_clean["AssociationFee_isna"] = df_clean["AssociationFee"].isna().astype(int)
    df_clean["AssociationFee"] = df_clean["AssociationFee"].fillna(0.0)

if "AssociationFeeFrequency" in df_clean.columns:
    df_clean["AssociationFeeFrequency"] = df_clean["AssociationFeeFrequency"].fillna("None")

In [46]:
#5. Missing value analysis check

# Total missing values per column
missing_count = df_clean.isnull().sum()

# Percentage of missing values per column
missing_percent = (df_clean.isnull().sum() / len(df_clean)) * 100

# Summary table of columns with missing values
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Percent Missing (%)': missing_percent
}).query('`Missing Count` > 0').sort_values(by='Missing Count', ascending=False)

print(missing_summary)
print(missing_summary.iloc[0:30])

                          Missing Count  Percent Missing (%)
ElementarySchool                 259531            86.085644
MiddleOrJuniorSchool             259215            85.980828
HighSchool                       246619            81.802773
SubdivisionName                  199752            66.257131
MainLevelBedrooms                118070            39.163460
Flooring                         105671            35.050750
HighSchoolDistrict                74455            24.696497
BuyerAgentAOR                     44301            14.694507
MLSAreaMajor                      43581            14.455685
ListAgentAOR                      41153            13.650325
Stories                           37596            12.470479
AttachedGarageYN                  33545            11.126775
PoolPrivateYN                     29474             9.776436
Levels                            28934             9.597320
ViewYN                            27432             9.099111
NewConstructionYN       

In [ ]:
from sklearn.preprocessing import TargetEncoder

# 6. Chronological Train / Validation / Test Split
def get_chronological_splits(data, train_months=6):
    """
    Carves out train, validation (2nd most recent month), and test (most recent month)
    without relying on external global variables.
    """
    latest_date = data['CloseDate'].max()
    test_start = latest_date.to_period('M').to_timestamp()                       # 1st day of most recent month
    val_start = (test_start - pd.DateOffset(months=1)).to_period('M').to_timestamp()  # 1st day of 2nd most recent month
    train_start = val_start - pd.DateOffset(months=train_months)
    
    train_data = data[(data['CloseDate'] >= train_start) & (data['CloseDate'] < val_start)].copy()
    val_data = data[(data['CloseDate'] >= val_start) & (data['CloseDate'] < test_start)].copy()
    test_data = data[data['CloseDate'] >= test_start].copy()
    
    return train_data, val_data, test_data

train_df, val_df, test_df = get_chronological_splits(df_clean, train_months=6)

# 7. Outlier Filtering Learned from Train Only
# Percentiles are computed ONLY on training data and applied frozen to val/test
p_05 = train_df['ClosePrice'].quantile(0.005)
p_995 = train_df['ClosePrice'].quantile(0.995)

train_clean = train_df[(train_df['ClosePrice'] >= p_05) & (train_df['ClosePrice'] <= p_995)].copy()
val_clean = val_df[(val_df['ClosePrice'] >= p_05) & (val_df['ClosePrice'] <= p_995)].copy()
test_clean = test_df[(test_df['ClosePrice'] >= p_05) & (test_df['ClosePrice'] <= p_995)].copy()

# 8. Preprocessing Pipeline
# Intrinsic property features free of target leakage 
# Group columns by data type and cardinality
num_features = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 
    'LotSizeSquareFeet', 'YearBuilt', 'AssociationFee', 
    'BelowGradeFinishedArea', 'AssociationFee_isna'
]
num_features = [c for c in num_features if c in df_clean.columns]

low_card_cat_features = [
    c for c in ['WaterfrontYN', 'BasementYN', 'AssociationFeeFrequency', 'PropertyCondition']
    if c in df_clean.columns
]

high_card_cat_features = [
    c for c in ['ElementarySchool','MiddleOrJuniorSchool', 'HighSchool','SubdivisionName', 'HighSchoolDistrict', 'BuyerAgentAOR', 'City', 'PostalCode']
    if c in df_clean.columns
]

# Define pipelines per feature type
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

low_card_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

high_card_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target_enc', TargetEncoder(smooth='auto', cv=5))
])

# Combine into a single preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('low_card', low_card_transformer, low_card_cat_features),
        ('high_card', high_card_transformer, high_card_cat_features)
    ],
    remainder='drop'
)

# Fit ONLY on training data, transform validation and test sets
X_train_processed = preprocessor.fit_transform(train_clean, train_clean['ClosePrice'])
X_val_processed = preprocessor.transform(val_clean)
X_test_processed = preprocessor.transform(test_clean)

print(f"Train set shape: {X_train_processed.shape}")
print(f"Validation set shape: {X_val_processed.shape}")
print(f"Test set shape: {X_test_processed.shape}")

Train set shape: (57866, 27)
Validation set shape: (11140, 27)
Test set shape: (12001, 27)
Total set shape: (301480, 61)
